# 美国 SEC 财报 · 简单解析

对**已下载**的 SEC HTML filing 做解析：读文件 → 解析成 soup → 抽元数据 → 按 10-K 分节。

**流程：** ① 路径与依赖 → ② 选择文件 → ③ 读取 HTML → ④ BeautifulSoup 解析 → ⑤ 元数据与分节（定义函数）→ ⑥ 运行并查看结果。  
数据来源：`02_US_SEC_report/data/filings/`（需先用 `download_filings.py` 下载）。

---
## 步骤 1：路径与依赖

从当前工作目录向上查找 `02_US_SEC_report`，加入 Python 路径并导入 `config`，得到财报目录 `FILINGS_DIR`。依赖：`beautifulsoup4`（未装则 `pip install beautifulsoup4`）。

In [11]:
import os
import sys

_cwd = os.path.abspath(os.getcwd())
_searched = _cwd
_sec_dir = None
for _ in range(10):
    _candidate = os.path.join(_searched, "02_US_SEC_report")
    if os.path.isdir(_candidate):
        _sec_dir = _candidate
        break
    _parent = os.path.dirname(_searched)
    if _parent == _searched:
        break
    _searched = _parent

if _sec_dir is None:
    raise FileNotFoundError(f"找不到 02_US_SEC_report。当前目录: {_cwd}")

sys.path.insert(0, _sec_dir)
from config import FILINGS_DIR, BASE_DIR

print("FILINGS_DIR:", FILINGS_DIR)
print("存在?", os.path.isdir(FILINGS_DIR))

FILINGS_DIR: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/02_US_SEC_report/data/filings
存在? True


---
## 步骤 2：输出目录与待处理文件

解析结果 JSON 输出到 **`US/parsed_filings/`**。待处理为 **Intel 10-K**、**NVDA 10-K** 各一份；步骤 3/4 用第一个文件做单文件预览。

In [12]:
# 解析结果 JSON 输出到 US 下的 parsed_filings 文件夹
OUTPUT_DIR = os.path.join(os.getcwd(), "parsed_filings")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("输出目录:", OUTPUT_DIR)

# 待处理：Intel 10-K、NVDA 10-K 各一份
FILES_TO_PROCESS = [
    ("INTEL CORP", "10-K_2024-01-26_intc-20231230.htm"),
    ("NVIDIA CORP", "10-K_2024-02-21_nvda-20240128.htm"),
]

# 下面用于步骤 3/4 单文件预览时使用第一个文件
company_folder, filename = FILES_TO_PROCESS[0]
file_path = os.path.join(FILINGS_DIR, company_folder, filename)
print("当前预览文件:", file_path)
print("存在?", os.path.exists(file_path))

输出目录: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/US/parsed_filings
当前预览文件: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/02_US_SEC_report/data/filings/INTEL CORP/10-K_2024-01-26_intc-20231230.htm
存在? True


---
## 步骤 3：读取 HTML

用 `open(..., encoding="utf-8")` 将文件读成字符串。SEC 报表一般为 UTF-8。

In [13]:
with open(file_path, "r", encoding="utf-8") as f:
    html_content = f.read()

print("字符数:", len(html_content))
print("前 200 字:", html_content[:200])

字符数: 3276073
前 200 字: <?xml version="1.0" ?><!--XBRL Document Created with the Workiva Platform--><!--Copyright 2024 Workiva--><!--r:f75179cb-2455-4e35-9e22-fcc52567cb6d,g:0c6cbb6c-a62c-49be-9a73-6e8cc6653c7b,d:df7ed2de495


---
## 步骤 4：BeautifulSoup 解析

将 HTML 字符串交给 BeautifulSoup，得到树形结构 `soup`。大文件建议安装 lxml（`pip install lxml`），解析会快很多；分节时用原始 HTML 字符串按 id 截取，避免逐节点遍历导致卡死。

In [14]:
from bs4 import BeautifulSoup

# 只解析前 800KB 用于 TOC 和元数据，避免大 10-K 卡死；分节时用完整 html_content 做字符串截取
MAX_PARSE_CHARS = 800000
html_for_soup = html_content[:MAX_PARSE_CHARS] if len(html_content) > MAX_PARSE_CHARS else html_content
if len(html_content) > MAX_PARSE_CHARS:
    print("文件较大，仅解析前", MAX_PARSE_CHARS // 1000, "KB 用于目录与元数据")

try:
    soup = BeautifulSoup(html_for_soup, "lxml")
    print("解析器: lxml")
except Exception:
    soup = BeautifulSoup(html_for_soup, "html.parser")
    print("解析器: html.parser")
print("类型:", type(soup))
print("title:", soup.title.string if soup.title else None)

文件较大，仅解析前 800 KB 用于目录与元数据
解析器: lxml
类型: <class 'bs4.BeautifulSoup'>
title: intc-20231230


/var/folders/nw/qn0916dd3g12_8kqhgfzzgp40000gn/T/ipykernel_42181/1948801298.py:10: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  soup = BeautifulSoup(html_for_soup, "lxml")


---
## 步骤 5：元数据与分节（定义函数）

**第一步·元数据**：从 soup 取 title、公司名（EntityRegistrantName）、文档类型（DocumentType）、财年（FiscalYearFocus 或正文正则）、总页数（estimate_page_count：在 table 里找 Item 16 / Form 10-K Summary 对应页码）。  
**第二步·分节**：`get_sections(soup, html_content)` 先按表格找目录：若有标准「Item 1」「Item 1A」等则用；**若无**（如 Intel 自定义目录），则从含 “contents” 的表格里收集所有 `<a href="#id">`。若链接内是**纯数字**（如页码 2、37），则用**该行整行文字**作 section 名，否则用链接文字。再在原始 HTML 里按 id 截取、去标签并清洗。  
**打包**：`extract_metadata(ticker, soup, filing_date)` 返回 `meta_data`、`sections`、`full_text`、`processing_info`。

In [15]:
import re
from datetime import datetime

def _normalize(text):
    """把 \xa0 等非标准空白统一成普通空格。"""
    return text.replace("\xa0", " ").replace("\u00a0", " ")

def estimate_page_count(soup):
    """在 table 里找 Item 16 / Form 10-K Summary 对应页码。"""
    for table in soup.find_all("table"):
        row_text = _normalize(table.get_text(separator=" ", strip=True)).lower()
        if "item 16" not in row_text and "form 10-k summary" not in row_text:
            continue
        for cell in table.find_all(["td", "th"]):
            t = cell.get_text(strip=True)
            if t.isdigit() and 1 <= int(t) <= 5000:
                return int(t)
    return None

# 长 key 在前，防止 "item 1" 抢匹配 "item 1a" 的行；\s+ 兼容 \xa0 归一化后的空格
ITEM_PATTERNS_10K = [
    ("item 1a",  re.compile(r"item\s+1\s*a\b", re.I)),
    ("item 1b",  re.compile(r"item\s+1\s*b\b", re.I)),
    ("item 1c",  re.compile(r"item\s+1\s*c\b", re.I)),
    ("item 1",   re.compile(r"item\s+1\b(?!\s*[a-z0-9])", re.I)),
    ("item 2",   re.compile(r"item\s+2\b", re.I)),
    ("item 3",   re.compile(r"item\s+3\b", re.I)),
    ("item 4",   re.compile(r"item\s+4\b", re.I)),
    ("item 5",   re.compile(r"item\s+5\b", re.I)),
    ("item 6",   re.compile(r"item\s+6\b", re.I)),
    ("item 7a",  re.compile(r"item\s+7\s*a\b", re.I)),
    ("item 7",   re.compile(r"item\s+7\b(?!\s*[a-z0-9])", re.I)),
    ("item 8",   re.compile(r"item\s+8\b", re.I)),
    ("item 9a",  re.compile(r"item\s+9\s*a\b", re.I)),
    ("item 9b",  re.compile(r"item\s+9\s*b\b", re.I)),
    ("item 9c",  re.compile(r"item\s+9\s*c\b", re.I)),
    ("item 9",   re.compile(r"item\s+9\b(?!\s*[a-z0-9])", re.I)),
    ("item 10",  re.compile(r"item\s+10\b", re.I)),
    ("item 11",  re.compile(r"item\s+11\b", re.I)),
    ("item 12",  re.compile(r"item\s+12\b", re.I)),
    ("item 13",  re.compile(r"item\s+13\b", re.I)),
    ("item 14",  re.compile(r"item\s+14\b", re.I)),
    ("item 15",  re.compile(r"item\s+15\b", re.I)),
    ("item 16",  re.compile(r"item\s+16\b", re.I)),
]
ITEM_KEY_ORDER = [k for k, _ in ITEM_PATTERNS_10K]

def _clean_section_text(text, max_chars=800000):
    """清洗节内文本；超长时只处理前 max_chars 字符，避免卡死。"""
    if not text or not isinstance(text, str): return ""
    t = text[:max_chars] if len(text) > max_chars else text
    t = _normalize(t)
    t = re.sub(r"\bPage\s+\d+\b", "", t, flags=re.IGNORECASE)
    t = re.sub(r"[^\n|]*\|\s*\d{4}\s*Form\s*10-K\|[^\n]*", "", t, flags=re.IGNORECASE)
    lines = [ln.strip() for ln in t.splitlines() if ln.strip()]
    seen = set()
    unique = []
    for ln in lines:
        if len(ln) > 2 and ln not in seen:
            seen.add(ln)
            unique.append(ln)
    return "\n".join(unique)

def _find_id_pos(html_str, sid, start=0):
    """在 html_str[start:] 里找 id=sid 第一次出现的位置；sid 可能是 id=\"sid\" 或 id='sid'。"""
    for quote in ('"', "'"):
        marker = 'id=' + quote + sid + quote
        p = html_str.find(marker, start)
        if p != -1:
            return p
    return -1

def _toc_from_tables(soup):
    """先从表格里找标准 Item 1..16；若无，则从任意表格收集「链接文字 → href#id」作为目录。"""
    toc_entries = {}
    max_possible = len(ITEM_PATTERNS_10K)
    for table in soup.find_all("table"):
        if len(toc_entries) >= max_possible:
            break
        for tr in table.find_all("tr"):
            row_lower = _normalize(tr.get_text(separator=" ", strip=True)).lower()
            for key, pattern in ITEM_PATTERNS_10K:
                if pattern.search(row_lower) and key not in toc_entries:
                    a = tr.find("a", href=re.compile(r"#.+"))
                    if a and a.get("href"):
                        toc_entries[key] = a["href"].lstrip("#").strip()
                    break
    if toc_entries:
        keys_ordered = [k for k in ITEM_KEY_ORDER if k in toc_entries]
        return keys_ordered, toc_entries
    # 备用：收集表格中带 href="#id" 的链接。若链接内是纯数字（如页码），用该行整行文字作 section 名
    candidates = []
    for table in soup.find_all("table"):
        links = []
        for a in table.find_all("a", href=re.compile(r"#.+")):
            href = a.get("href")
            if not href:
                continue
            sid = href.lstrip("#").strip()
            if not sid or len(sid) < 3:
                continue
            text = a.get_text(strip=True) or ""
            tr = a.find_parent("tr")
            if tr and (not text or text.isdigit() or len(text) <= 2):
                row_text = tr.get_text(separator=" ", strip=True)
                row_text = " ".join(row_text.split())[:80]
                key = row_text if row_text else ("section_%d" % (len(links) + 1))
            else:
                key = text[:100] if len(text) > 100 else (text or ("section_%d" % (len(links) + 1)))
            links.append((key, sid))
        if len(links) >= 3:
            has_contents = "contents" in table.get_text(separator=" ", strip=True).lower()
            candidates.append((has_contents, links))
    if candidates:
        candidates.sort(key=lambda x: (not x[0], -len(x[1])))
        links = candidates[0][1]
        keys_ordered = []
        toc_entries = {}
        for i, (k, sid) in enumerate(links):
            key = k if k not in toc_entries else (k + "_%d" % i)
            keys_ordered.append(key)
            toc_entries[key] = sid
        return keys_ordered, toc_entries
    return [], {}

def get_sections(soup, html_content=None):
    """找目录（标准 Item 1..16 或备用「链接→id」）→ 按 id 在原始 HTML 里截取每节、正则去标签 → 清洗。"""
    keys_ordered, toc_entries = _toc_from_tables(soup)
    if not toc_entries:
        body = soup.find("body")
        full = body.get_text(separator=" ", strip=True) if body else ""
        return {"sections": {}, "full_text": [_clean_section_text(full)], "total_sections": 0}
    html_str = html_content if html_content is not None else str(soup)
    sections = {}
    full_text = []
    for i, key in enumerate(keys_ordered):
        sid = toc_entries[key]
        next_id = toc_entries.get(keys_ordered[i + 1]) if i + 1 < len(keys_ordered) else None
        start_pos = _find_id_pos(html_str, sid)
        if start_pos == -1:
            sections[key] = ""
            full_text.append("")
            continue
        tag_end = html_str.find(">", start_pos)
        chunk_start = tag_end + 1 if tag_end != -1 else start_pos
        if next_id is None:
            chunk = html_str[chunk_start:]
        else:
            end_pos = _find_id_pos(html_str, next_id, chunk_start)
            if end_pos == -1:
                chunk = html_str[chunk_start:]
            else:
                chunk = html_str[chunk_start:end_pos]
        chunk = re.sub(r"<[^>]+>", " ", chunk)
        chunk = " ".join(chunk.split())
        section_text = _clean_section_text(chunk)
        sections[key] = section_text
        full_text.append(section_text)
    return {"sections": sections, "full_text": full_text, "total_sections": len(sections)}

def _get_meta(soup, name_contains):
    tag = soup.find(attrs={"name": lambda x: x and name_contains in str(x)})
    return tag.get_text(strip=True) if tag else None

def extract_metadata(ticker, soup, filing_date, html_content=None):
    """① 元数据 ② get_sections。传入 html_content 时用字符串截取分节（大文件更快）。返回 meta_data, sections, full_text, processing_info。"""
    title = soup.title.string if soup.title else None
    company = _get_meta(soup, "EntityRegistrantName")
    document_type = _get_meta(soup, "dei:DocumentType")
    fiscal_year = _get_meta(soup, "dei:DocumentFiscalYearFocus")
    if not fiscal_year:
        m = re.search(r"Fiscal\s+Year\s+Ended\s+[^\d]*(\d{4})\b", soup.get_text(separator=" ", strip=True), re.I)
        if m: fiscal_year = m.group(1)
    total_page = estimate_page_count(soup)
    meta_data = {"title": title, "ticker": ticker, "filing_date": filing_date, "Company": company, "document_type": document_type, "fiscal_year": fiscal_year, "total_page": total_page}
    sec = get_sections(soup, html_content)
    total_words = sum(len(t.split()) for t in sec["full_text"] if t)
    processing_info = {"parsed_at": datetime.now().isoformat(), "total_sections": sec["total_sections"], "total_words": total_words}
    return {"meta_data": meta_data, "sections": sec["sections"], "full_text": sec["full_text"], "processing_info": processing_info}

**Section 标题是怎么来的（分节逻辑简述）**

- **第一步：找目录（TOC）**  
  在 HTML 里扫所有 `<table>`，有两种方式二选一：

  1. **标准方式**：在表格的某一行里找是否出现 `"item 1"`、`"item 1a"`、…、`"item 16"` 这些固定字符串。  
     - 若找到：这一行的 **section 标题** 就取这些**固定 key**（如 `"item 1"`、`"item 1a"`）。  
     - 同时在该行里找一个 `<a href="#某个id">`，用 **`#` 后面的 id** 在正文里定位这一节的起止位置。

  2. **备用方式**：若没有任何表格包含上述 Item 文案（例如 Intel 那种自定义目录），就从「含有较多 `href="#..."` 链接」的表格里收集链接。  
     - 每个链接会得到：**锚点 id**（`href` 里 `#` 后面的部分，用来在 HTML 里定位）、以及 **section 标题**。  
     - **标题**的取法：  
       - 若 `<a>...</a>` 里的文字是**纯数字**或**很短**（≤2 字，多半是页码）：不用它，改用**该链接所在整行（`<tr>`）的文本**（截到 80 字）作为 section 标题。  
       - 否则：用 **`<a>` 里的文字**作为 section 标题。  
     - 若有多个表格都满足，优先选表格里出现 "contents" 的那个。

- **第二步：按标题切正文**  
  用上一步得到的「标题 → 锚点 id」列表，在原始 HTML 里找到每个 id 的位置，从当前 id 截到下一个 id 之前，去掉标签、清洗后得到该节的纯文本。  
  最终 `sections` 的 **key** 就是上面说的 **section 标题**（要么是 "item 1" 这种，要么是行文本/链接文字）。

---
## 步骤 6：运行并查看结果

从文件名解析出 `filing_date` 和 `ticker`，调用 `extract_metadata`，打印 `meta_data`、`processing_info` 和 `sections` 的 key。默认已选 10-K 文件，会得到按 Item 分好的 `sections`。

In [16]:
import time
parts = filename.replace(".htm", "").replace(".html", "").split("_")
filing_date = parts[1] if len(parts) >= 2 else ""
ticker = parts[2].split("-")[0].upper() if len(parts) >= 3 else ""

t0 = time.perf_counter()
final_data = extract_metadata(ticker, soup, filing_date, html_content)
print(f"解析耗时: {time.perf_counter() - t0:.1f} 秒\n")

print("=== meta_data ===")
for k, v in final_data["meta_data"].items():
    print(f"  {k}: {v}")
print("\n=== processing_info ===")
for k, v in final_data["processing_info"].items():
    print(f"  {k}: {v}")
print("\n=== sections 的 key ===")
print(list(final_data["sections"].keys())[:12] if final_data["sections"] else "（无 10-K 分节）")
print("\nfull_text 条数:", len(final_data["full_text"]))

解析耗时: 131.9 秒

=== meta_data ===
  title: intc-20231230
  ticker: INTC
  filing_date: 2024-01-26
  Company: INTEL CORPORATION
  document_type: 10-K
  fiscal_year: 2023
  total_page: None

=== processing_info ===
  parsed_at: 2026-03-16T22:52:26.109720
  total_sections: 29
  total_words: 72019

=== sections 的 key ===
['Availability of Company Information 2', 'Introduction to Our Business 3', 'A Year in Review 5', 'Our Strategy 7', 'Our Capital 10', 'Our Products 20', 'Segment Trends and Results 21', 'Consolidated Results of Operations 37', 'Liquidity and Capital Resources 42', 'Critical Accounting Estimates 44', 'Non-GAAP Financial Measures 45', 'Risk Factors 48']

full_text 条数: 29


---
## 小结

本 Notebook 完成：读 HTML → BeautifulSoup 解析 → 元数据抽取 → 按 10-K Item 分节与清洗 → 得到结构化结果（`meta_data`、`sections`、`full_text`、`processing_info`）。后续可在此基础上落库（companies / documents / document_sections）或做 RAG 等。

---
## 批量处理 Intel / NVDA 10-K 并保存到 parsed_filings

对 `FILES_TO_PROCESS` 中的两个 10-K 分别：读 HTML → 解析 → 抽元数据与分节 → 将结果保存为 `parsed_filings/{ticker}_10K_{filing_date}.json`。

In [17]:
# import json
# import time

# # 与步骤 4 一致：仅用前 800KB 建 soup，分节用完整 html_content
# MAX_PARSE_CHARS = 800000

# for company_folder, filename in FILES_TO_PROCESS:
#     path = os.path.join(FILINGS_DIR, company_folder, filename)
#     if not os.path.exists(path):
#         print(f"跳过（不存在）: {path}")
#         continue
#     with open(path, "r", encoding="utf-8") as f:
#         html_content = f.read()
#     html_for_soup = html_content[:MAX_PARSE_CHARS] if len(html_content) > MAX_PARSE_CHARS else html_content
#     try:
#         soup = BeautifulSoup(html_for_soup, "lxml")
#     except Exception:
#         soup = BeautifulSoup(html_for_soup, "html.parser")
#     parts = filename.replace(".htm", "").replace(".html", "").split("_")
#     filing_date = parts[1] if len(parts) >= 2 else ""
#     ticker = (parts[2].split("-")[0]).upper() if len(parts) >= 3 else ""
#     t0 = time.perf_counter()
#     final_data = extract_metadata(ticker, soup, filing_date, html_content)
#     elapsed = time.perf_counter() - t0
#     out_path = os.path.join(OUTPUT_DIR, f"{ticker}_10K_{filing_date}.json")
#     with open(out_path, "w", encoding="utf-8") as f:
#         json.dump(final_data, f, ensure_ascii=False, indent=2)
#     print(f"[{ticker}] 已保存: {out_path}  耗时 {elapsed:.1f}s")
# print("全部完成。")

---
## 批量处理所有公司 10-K 并保存到 parsed_filings

从 `02_US_SEC_report/data/filings` 下，自动找到所有公司文件夹里文件名以 `10-K_` 开头的 filing，逐个解析并保存到 `parsed_filings/`。文件名格式：`{TICKER}_10K_{filing_date}.json`。

In [ ]:
import json
import time
import signal
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

TIMEOUT_SECONDS = 20 * 60  # 20 分钟超时

class TimeoutError(Exception):
    pass

def _timeout_handler(signum, frame):
    raise TimeoutError("处理超时")

signal.signal(signal.SIGALRM, _timeout_handler)

try:
    MAX_PARSE_CHARS
except NameError:
    MAX_PARSE_CHARS = 800000

all_10k_files = []
for company_folder in sorted(os.listdir(FILINGS_DIR)):
    company_path = os.path.join(FILINGS_DIR, company_folder)
    if not os.path.isdir(company_path):
        continue
    for fname in sorted(os.listdir(company_path)):
        if not fname.startswith("10-K_"):
            continue
        if not (fname.endswith(".htm") or fname.endswith(".html")):
            continue
        all_10k_files.append((company_folder, fname))

print(f"共找到 10-K 文件: {len(all_10k_files)} 个")

skipped = 0
for idx, (company_folder, filename) in enumerate(all_10k_files, start=1):
    parts = filename.replace(".htm", "").replace(".html", "").split("_")
    filing_date = parts[1] if len(parts) >= 2 else ""
    ticker = (parts[2].split("-")[0]).upper() if len(parts) >= 3 else ""
    out_path = os.path.join(OUTPUT_DIR, f"{ticker}_10K_{filing_date}.json")

    if os.path.exists(out_path):
        skipped += 1
        continue

    path = os.path.join(FILINGS_DIR, company_folder, filename)
    if not os.path.exists(path):
        print(f"[{idx}/{len(all_10k_files)}] 跳过（不存在）: {path}")
        continue

    print(f"[{idx}/{len(all_10k_files)}] 开始处理: {ticker} {filing_date} ...", end=" ", flush=True)

    try:
        signal.alarm(TIMEOUT_SECONDS)

        with open(path, "r", encoding="utf-8") as f:
            html_content = f.read()

        html_for_soup = html_content[:MAX_PARSE_CHARS] if len(html_content) > MAX_PARSE_CHARS else html_content
        try:
            soup = BeautifulSoup(html_for_soup, "lxml")
        except Exception:
            soup = BeautifulSoup(html_for_soup, "html.parser")

        t0 = time.perf_counter()
        final_data = extract_metadata(ticker, soup, filing_date, html_content)
        elapsed = time.perf_counter() - t0

        signal.alarm(0)

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(final_data, f, ensure_ascii=False, indent=2)

        n_sec = final_data["processing_info"]["total_sections"]
        print(f"OK  {n_sec} sections  {elapsed:.1f}s")

    except TimeoutError:
        signal.alarm(0)
        print(f"SKIP  超过 {TIMEOUT_SECONDS // 60} 分钟，跳过")
    except Exception as e:
        signal.alarm(0)
        print(f"ERROR  {e}")

if skipped:
    print(f"\n跳过已存在: {skipped} 个")
print(f"全部 10-K 处理完成。")

共找到 10-K 文件: 118 个
[11/118] 开始处理: AXP 2025-02-07 ... 

SKIP  超过 20 分钟，跳过
[12/118] 开始处理: AXP 2026-02-06 ... SKIP  超过 20 分钟，跳过
[13/118] 开始处理: ABBV 2022-02-18 ... OK  19 sections  23.5s
[14/118] 开始处理: ABBV 2023-02-17 ... OK  22 sections  28.2s
[15/118] 开始处理: ABBV 2024-02-20 ... OK  22 sections  29.4s
[16/118] 开始处理: ABBV 2025-02-14 ... OK  23 sections  31.6s
[17/118] 开始处理: GOOG 2023-02-03 ... OK  22 sections  591.1s
[18/118] 开始处理: GOOG 2024-01-31 ... OK  23 sections  650.6s
[19/118] 开始处理: GOOG 2025-02-05 ... OK  23 sections  721.3s
[20/118] 开始处理: GOOG 2026-02-05 ... OK  23 sections  733.4s
[21/118] 开始处理: AAPL 2020-10-30 ... OK  21 sections  15.0s
[22/118] 开始处理: AAPL 2021-10-29 ... OK  22 sections  13.2s
[23/118] 开始处理: AAPL 2022-10-28 ... OK  22 sections  12.3s
[24/118] 开始处理: AAPL 2023-11-03 ... OK  23 sections  10.8s
[25/118] 开始处理: AAPL 2024-11-01 ... OK  23 sections  11.3s
[26/118] 开始处理: AAPL 2025-10-31 ... OK  23 sections  11.3s
[27/118] 开始处理: AVGO 2020-12-18 ... SKIP  超过 20 分钟，跳过
[28/118] 开始处理: AVGO 2021-12-17 ... OK  21 sections  891.0s
[2